# Build Your First RAG Pipeline
With **Youssef Bastawisy**


### **Step 0: Setup & Installations**

First, we need to install the necessary libraries.

*  LangChain: The framework that ties everything together.
*  ChromaDB: Our Vector Database.
*  Sentence-Transformers & HuggingFace: To create our mathematical embeddings.
*  Groq: To access Llama-3 for free and at lightning speed.
*  PyPDF: To read PDF files.















In [ ]:
# Install required libraries
!pip install -qU langchain langchain-community langchain-core
!pip install -qU langchain-huggingface langchain-groq
!pip install -qU chromadb pypdf sentence-transformers

print("✅ Installations complete!")


### **Step 1: API Keys**

To use the LLM (Llama 3), we need a free Groq API key.

Go to [console.groq.com](https://console.groq.com)

Create an account and generate an API key.

Click on the 🔑 "Secrets" icon on the left sidebar of Google Colab, add a new secret named `GROQ_API_KEY`, and paste your key.

In [ ]:
import os
from google.colab import userdata

# Safely load the API key from Colab Secrets
os.environ["GROQ_API_KEY"] = userdata.get('GROQ_API_KEY')

print("✅ API Key loaded successfully!")

### **Step 2: Ingestion (Loading our Data)**

Let's download a sample document to act as our "External Brain." For this tutorial, we will download the famous AI paper "Attention Is All You Need".

In [ ]:
# Download a sample PDF directly to Colab
!wget -qO sample_document.pdf [https://arxiv.org/pdf/1706.03762.pdf](https://arxiv.org/pdf/1706.03762.pdf)

from langchain_community.document_loaders import PyPDFLoader

# Load the PDF
loader = PyPDFLoader("sample_document.pdf")
documents = loader.load()

print(f"✅ Loaded {len(documents)} pages from the PDF.")
# Let's peek at the first 200 characters of page 1
print("\nPreview:\n", documents[0].page_content[:200])

### **Step 3: Chunking the Data**

LLMs have a context window (memory limit). We can't feed the whole document at once. We need to split it into smaller, overlapping chunks.

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Create the splitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,       # Number of characters per chunk
    chunk_overlap=100,     # Overlap to prevent cutting sentences in half
    length_function=len
)

# Split the document
chunks = text_splitter.split_documents(documents)

print(f"✅ Split the document into {len(chunks)} chunks.")
print(f"Example chunk length: {len(chunks[0].page_content)} characters.")


### **Step 4: Embeddings & Vector Database**

Now we turn our text chunks into arrays of numbers (vectors) and store them in ChromaDB so we can search through them mathematically. We will use a free, open-source embedding model from HuggingFace.

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

# Initialize the embedding model
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

# Create the Vector Database and embed all chunks
print("Embedding chunks and building Vector DB... (This might take a minute)")
vector_db = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory="./chroma_db"
)

print("✅ Vector Database created successfully!")


### **Step 5: Setting up the LLM & The Retriever**

We'll set up Llama-3 (via Groq) as our brain, and convert our vector database into a "Retriever" that searches for the top 3 most relevant chunks when asked a question.

In [ ]:
from langchain_groq import ChatGroq

# Initialize the LLM (Llama 3 8B)
llm = ChatGroq(
    model="llama3-8b-8192",
    temperature=0.2 # Low temperature for more factual, less creative answers
)

# Convert our Vector DB into a Retriever (Fetch top 3 results)
retriever = vector_db.as_retriever(search_kwargs={"k": 3})

print("✅ LLM and Retriever are ready!")

### **Step 6: Building the RAG Chain**

This is where we connect the dots. We create a strict prompt that forces the LLM to only use the retrieved context to answer the question.

In [ ]:
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

# Define the Prompt Template
system_prompt = (
    "You are a helpful AI assistant for Computer science students. "
    "Use the following pieces of retrieved context to answer the question. "
    "If you don't know the answer, just say that you don't know. "
    "Use three sentences maximum and keep the answer concise.\n\n"
    "Context:\n{context}"
)

prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{input}"),
])

# Create the chain that combines the documents and sends them to the LLM
question_answer_chain = create_stuff_documents_chain(llm, prompt)

# Combine the Retriever and the LLM chain
rag_chain = create_retrieval_chain(retriever, question_answer_chain)

print("✅ RAG Chain assembled!")

### **Step 7: Testing Time!**

Let's ask our RAG system a specific question about the paper we uploaded.

In [ ]:
# The question we want to ask
user_question = "What is the exact name of the architecture introduced in this paper, and what mechanism does it entirely rely on?"

# Run the RAG pipeline
response = rag_chain.invoke({"input": user_question})

print("👤 Question:", user_question)
print("\n🤖 RAG Answer:\n", response["answer"])

print("\n---")
print("🔍 Where did the LLM get this information?")
# Let's look at the exact chunks it retrieved to formulate this answer
for i, doc in enumerate(response["context"]):
    print(f"\nSource {i+1} (Page {doc.metadata.get('page', 'Unknown')}):")
    print(doc.page_content[:200] + "...")